# EfficientNet-B0 Eyebrow ROI Baseline — Nazlıcan / Deney 1

Bu notebook, verilen **EfficientNet-B0 göz ROI baseline** kodunun kaş bölgesine teknik olarak uyarlanmış sürümüdür. Sadece isim/path değişikliği yapılmamıştır; kaş ROI'lerinin geniş ve kısa geometrisini bozmamak için **aspect-ratio koruyan resize + padding** kullanılır.

**Sabit proje yolları**

- Input root: `MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Kaş`
- Metadata: `Kaş/metadata.csv`
- ROI images: `Kaş/{real|fake}/{train|val|test}/...`
- Output: `MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar/<run_id>`

**Çıktı klasör standardı**

Her run kendi klasörüne yazılır ve şu yapıyı üretir:

- `checkpoints/` → `best.ckpt`, `last.ckpt`, son 3 `epoch_N.ckpt`
- `logs/`
- `metrics/`
- `predictions/`
- `figures/` → İngilizce başlık/eksen/legend, kısa kenar ≥ 600 px
- `artifacts/`
- `config_resolved.yaml`
- `requirements_lock.txt`
- `environment.json`
- `run_summary.json`
- `output_manifest.csv`

**Eğitim politikası**

- Mevcut `train / val / test` klasörleri **yeniden frame düzeyinde bölünmez**; upstream split korunur.
- `SUCCESS` ROI kayıtları eğitime alınır; `SKIPPED/ERROR` kayıtları audit için saklanır.
- Google Drive görselleri training sırasında batch batch okunmaz; önce Colab lokal SSD'sine cache edilir.
- Stage 1: pretrained EfficientNet-B0 backbone frozen, classifier eğitilir.
- Stage 2: bütün ağ düşük öğrenme oranıyla fine-tune edilir.
- Threshold yalnızca validation setinden seçilir.
- Test seti yalnızca final değerlendirmede kullanılır.
- Eğitim FP32 çalışır; NaN/Inf kalite kapıları aktiftir.
- Checkpointler atomik kaydedilir ve resume için optimizer/scheduler/RNG durumu tutulur.

> **Önemli veri-provenance notu:** Mevcut `Kaş/metadata.csv` dosyasında `source_video` alanı boşsa notebook gerçek video/kişi bazlı split izolasyonunu uydurmaz. Exact path/hash leakage kontrollerini yapar ve durumu `NOT_VERIFIABLE` olarak raporlar. Tam SSOT uyumu için upstream metadata'da gerçek `source_video` veya kişi ID bulunmalıdır.


In [1]:
# ============================================================
# CELL 1 — IMPORTS
# ============================================================

import os
import gc
import sys
import json
import math
import time
import random
import shutil
import hashlib
import platform
import subprocess
import logging
from dataclasses import dataclass, asdict
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any

import numpy as np
import pandas as pd
import yaml
from PIL import Image, ImageFile

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = False

print("Python      :", platform.python_version())
print("PyTorch     :", torch.__version__)
print("CUDA        :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU         :", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)


Python      : 3.12.13
PyTorch     : 2.11.0+cu128
CUDA        : True
GPU         : Tesla T4
CUDA version: 12.8


In [2]:
# ============================================================
# CELL 2 — CONFIGURATION
# ============================================================

RUN_SCHEMA_VERSION = 5

@dataclass(frozen=True)
class Config:
    seed: int = 42
    image_size: int = 224

    batch_size: int = 32
    num_workers: int = 0

    frozen_epochs: int = 5
    finetune_epochs: int = 15

    frozen_lr: float = 1e-3
    finetune_lr: float = 2e-5
    weight_decay: float = 1e-4

    early_stopping_patience: int = 4
    scheduler_patience: int = 2
    grad_clip_norm: float = 1.0
    epoch_checkpoint_keep: int = 3

    dropout: float = 0.2

    threshold_min: float = 0.05
    threshold_max: float = 0.95
    threshold_steps: int = 181

    accepted_statuses: Tuple[str, ...] = ("ok", "success")
    negative_label: str = "real"
    positive_label: str = "fake"

    cache_images_locally: bool = True
    verify_cached_images: bool = True

    resume_compatible_run: bool = True

    # Current eyebrow metadata has no authoritative source_video values.
    # True = train anyway, but record the split provenance as NOT_VERIFIABLE.
    # Set False when you want strict SSOT behavior that blocks training
    # until source_video/person_id exists.
    allow_unverifiable_source_split: bool = True

CONFIG = Config()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(CONFIG)
print("DEVICE:", DEVICE)


Config(seed=42, image_size=224, batch_size=32, num_workers=0, frozen_epochs=5, finetune_epochs=15, frozen_lr=0.001, finetune_lr=2e-05, weight_decay=0.0001, early_stopping_patience=4, scheduler_patience=2, grad_clip_norm=1.0, epoch_checkpoint_keep=3, dropout=0.2, threshold_min=0.05, threshold_max=0.95, threshold_steps=181, accepted_statuses=('ok', 'success'), negative_label='real', positive_label='fake', cache_images_locally=True, verify_cached_images=True, resume_compatible_run=True, allow_unverifiable_source_split=True)
DEVICE: cuda


In [3]:

# ============================================================
# CELL 3 — REPRODUCIBILITY
# ============================================================

def seed_everything(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    # Reproducibility first. warn_only avoids hard failures on unsupported ops.
    torch.use_deterministic_algorithms(True, warn_only=True)

    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True

seed_everything(CONFIG.seed)

print("Global seed fixed:", CONFIG.seed)


Global seed fixed: 42


In [4]:

# ============================================================
# CELL 4 — MOUNT GOOGLE DRIVE
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError as e:
    raise RuntimeError(
        "This notebook is intended for Google Colab."
    ) from e

MY_DRIVE = Path("/content/drive/MyDrive")

if not MY_DRIVE.is_dir():
    raise RuntimeError("Google Drive mount failed.")

print("Google Drive mounted:", MY_DRIVE)


Mounted at /content/drive
Google Drive mounted: /content/drive/MyDrive


In [5]:
# ============================================================
# CELL 5 — FIXED PROJECT PATHS (NAZLICAN / EYEBROW)
# ============================================================

DENEY1_ROOT = (
    MY_DRIVE
    / "AISC DeepFake Çalışmaları"
    / "Deneyler"
    / "Nazlıcan"
    / "Deney 1"
)

KAS_ROOT = DENEY1_ROOT / "Kaş"
METADATA_PATH = KAS_ROOT / "metadata.csv"

# Outputs go ONLY here. The Kaş input folder is read-only for this notebook.
RESULTS_ROOT = DENEY1_ROOT / "Sonuçlar"

required_directories = {
    "DENEY1_ROOT": DENEY1_ROOT,
    "KAS_ROOT": KAS_ROOT,
    "REAL_ROOT": KAS_ROOT / "real",
    "FAKE_ROOT": KAS_ROOT / "fake",
}

for name, path in required_directories.items():
    if not path.is_dir():
        raise FileNotFoundError(
            f"{name} was not found:\n{path}"
        )

if not METADATA_PATH.is_file():
    raise FileNotFoundError(
        f"metadata.csv was not found:\n{METADATA_PATH}"
    )

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("PATH CONFIGURATION PASSED")
print("=" * 80)
print("DENEY1_ROOT :", DENEY1_ROOT)
print("KAS_ROOT    :", KAS_ROOT)
print("METADATA    :", METADATA_PATH)
print("RESULTS_ROOT:", RESULTS_ROOT)


PATH CONFIGURATION PASSED
DENEY1_ROOT : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1
KAS_ROOT    : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Kaş
METADATA    : /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Kaş/metadata.csv
RESULTS_ROOT: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar


In [6]:
# ============================================================
# CELL 6 — RUN DIRECTORY + SAFE RESUME POLICY
# ============================================================

# SSOT run-id format:
# YYYYMMDD_HHMM_<region>_<model>_seed<N>
RUN_SUFFIX = "eyebrow_efficientnet_b0_seed42"
RUN_SCHEMA_FILE = "run_schema.json"
RUN_COMPLETE_FILE = "RUN_COMPLETE.json"

def is_compatible_incomplete_run(path: Path) -> bool:
    schema_path = path / RUN_SCHEMA_FILE
    complete_path = path / RUN_COMPLETE_FILE

    if not path.is_dir():
        return False

    if complete_path.exists():
        return False

    if not schema_path.is_file():
        return False

    try:
        with open(schema_path, "r", encoding="utf-8") as f:
            schema = json.load(f)

        return (
            int(schema.get("schema_version", -1)) == RUN_SCHEMA_VERSION
            and schema.get("run_suffix") == RUN_SUFFIX
        )
    except Exception:
        return False


compatible_runs = []

if CONFIG.resume_compatible_run:
    for candidate in RESULTS_ROOT.glob(f"*_{RUN_SUFFIX}*"):
        if is_compatible_incomplete_run(candidate):
            compatible_runs.append(candidate)

compatible_runs.sort(
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)

if compatible_runs:
    RUN_DIR = compatible_runs[0]
    RUN_ID = RUN_DIR.name
    print("Resuming compatible incomplete run:")
    print(RUN_DIR)
else:
    RUN_ID = (
        datetime.now().strftime("%Y%m%d_%H%M")
        + f"_{RUN_SUFFIX}"
    )

    RUN_DIR = RESULTS_ROOT / RUN_ID

    retry = 1

    while RUN_DIR.exists():
        RUN_DIR = RESULTS_ROOT / f"{RUN_ID}_r{retry}"
        retry += 1

    RUN_ID = RUN_DIR.name
    RUN_DIR.mkdir(parents=True, exist_ok=False)

    with open(
        RUN_DIR / RUN_SCHEMA_FILE,
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            {
                "schema_version": RUN_SCHEMA_VERSION,
                "run_id": RUN_ID,
                "run_suffix": RUN_SUFFIX,
                "created_at": datetime.now().isoformat(),
            },
            f,
            indent=2,
        )

DIRS = {
    "checkpoints": RUN_DIR / "checkpoints",
    "metrics": RUN_DIR / "metrics",
    "predictions": RUN_DIR / "predictions",
    "figures": RUN_DIR / "figures",
    "artifacts": RUN_DIR / "artifacts",
    "logs": RUN_DIR / "logs",
}

for directory in DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)

FROZEN_DIR = DIRS["checkpoints"] / "frozen"
FINETUNE_DIR = DIRS["checkpoints"] / "finetune"

FROZEN_DIR.mkdir(parents=True, exist_ok=True)
FINETUNE_DIR.mkdir(parents=True, exist_ok=True)

print("RUN_ID :", RUN_ID)
print("RUN_DIR:", RUN_DIR)


RUN_ID : 20260808_1248_eyebrow_efficientnet_b0_seed42
RUN_DIR: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar/20260808_1248_eyebrow_efficientnet_b0_seed42


In [7]:
# ============================================================
# CELL 7 — ATOMIC I/O
# ============================================================

def atomic_write_bytes(data: bytes, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)

    tmp = target.with_suffix(
        target.suffix + ".tmp"
    )

    with open(tmp, "wb") as f:
        f.write(data)
        f.flush()
        os.fsync(f.fileno())

    os.replace(tmp, target)


def atomic_write_text(
    text: str,
    target: Path,
    encoding: str = "utf-8",
) -> None:
    atomic_write_bytes(
        text.encode(encoding),
        target,
    )


def atomic_json_dump(
    obj: Any,
    target: Path,
) -> None:
    atomic_write_text(
        json.dumps(
            obj,
            indent=2,
            ensure_ascii=False,
            default=str,
        ),
        target,
    )


def atomic_yaml_dump(
    obj: Any,
    target: Path,
) -> None:
    atomic_write_text(
        yaml.safe_dump(
            obj,
            sort_keys=False,
            allow_unicode=True,
        ),
        target,
    )


def atomic_csv_dump(
    df: pd.DataFrame,
    target: Path,
) -> None:
    atomic_write_bytes(
        df.to_csv(index=False).encode("utf-8"),
        target,
    )


def atomic_torch_save(
    state: dict,
    target: Path,
) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)

    tmp = target.with_suffix(
        target.suffix + ".tmp"
    )

    torch.save(
        state,
        tmp,
    )

    # Read-back validation on CPU.
    verify = torch.load(
        tmp,
        map_location="cpu",
        weights_only=False,
    )

    required_keys = {
        "checkpoint_schema_version",
        "epoch",
        "model_state_dict",
        "optimizer_state_dict",
        "stage_name",
    }

    if not required_keys.issubset(
        verify.keys()
    ):
        try:
            tmp.unlink()
        finally:
            raise RuntimeError(
                f"Checkpoint validation failed: {tmp}"
            )

    os.replace(
        tmp,
        target,
    )


def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)

            if not chunk:
                break

            h.update(chunk)

    return h.hexdigest()


atomic_yaml_dump(
    asdict(CONFIG),
    RUN_DIR / "config_resolved.yaml",
)

atomic_json_dump(
    {
        "run_id": RUN_ID,
        "schema_version": RUN_SCHEMA_VERSION,
        "python": platform.python_version(),
        "torch": torch.__version__,
        "device": str(DEVICE),
        "gpu": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),
        "platform": platform.platform(),
    },
    RUN_DIR / "environment.json",
)

try:
    freeze = subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"],
        text=True,
    )

    atomic_write_text(
        freeze,
        RUN_DIR / "requirements_lock.txt",
    )
except Exception as e:
    atomic_write_text(
        f"pip freeze failed: {repr(e)}\n",
        RUN_DIR / "requirements_lock.txt",
    )


LOGGER = logging.getLogger(f"eyebrow_efficientnet_{RUN_ID}")
LOGGER.setLevel(logging.INFO)
LOGGER.handlers.clear()

_file_handler = logging.FileHandler(
    DIRS["logs"] / "training.log",
    mode="a",
    encoding="utf-8",
)
_file_handler.setFormatter(
    logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
)
LOGGER.addHandler(_file_handler)
LOGGER.propagate = False

LOGGER.info("Run initialized: %s", RUN_ID)
LOGGER.info("Input eyebrow root: %s", KAS_ROOT)
LOGGER.info("Results root: %s", RESULTS_ROOT)

print("Atomic I/O helpers + file logger ready.")


Atomic I/O helpers + file logger ready.


In [8]:
# ============================================================
# CELL 8 — LOAD + VALIDATE EYEBROW ROI METADATA
# ============================================================

REQUIRED_METADATA_COLUMNS = {
    "sample_id",
    "label",
    "split",
    "status",
}

metadata_raw = pd.read_csv(
    METADATA_PATH,
    encoding="utf-8-sig",
)

metadata_raw.columns = [
    str(column).strip()
    for column in metadata_raw.columns
]

missing_columns = REQUIRED_METADATA_COLUMNS.difference(
    metadata_raw.columns
)

if missing_columns:
    raise ValueError(
        "metadata.csv is missing required columns: "
        f"{sorted(missing_columns)}"
    )

if (
    "output_relative_path" not in metadata_raw.columns
    and "output_path" not in metadata_raw.columns
):
    raise ValueError(
        "metadata.csv must contain output_relative_path or output_path."
    )

metadata = metadata_raw.copy()

text_columns = [
    column
    for column in [
        "sample_id",
        "label",
        "split",
        "status",
        "skip_reason",
        "source_video",
        "output_relative_path",
        "output_path",
        "output_sha256",
        "input_sha256",
    ]
    if column in metadata.columns
]

for column in text_columns:
    metadata[column] = (
        metadata[column]
        .astype("string")
        .str.strip()
    )

metadata["label"] = metadata["label"].str.lower()
metadata["split"] = metadata["split"].str.lower()
metadata["status"] = metadata["status"].str.lower()

allowed_labels = {
    CONFIG.negative_label,
    CONFIG.positive_label,
}

allowed_splits = {
    "train",
    "val",
    "test",
}

accepted_statuses = {
    str(value).lower()
    for value in CONFIG.accepted_statuses
}

known_statuses = accepted_statuses | {
    "skipped",
    "error",
}

observed_statuses = set(
    metadata["status"]
    .dropna()
    .unique()
)

unknown_statuses = sorted(
    observed_statuses
    - known_statuses
)

if unknown_statuses:
    raise ValueError(
        f"Unexpected metadata statuses: {unknown_statuses}"
    )

observed_labels = set(
    metadata["label"]
    .dropna()
    .unique()
)

observed_splits = set(
    metadata["split"]
    .dropna()
    .unique()
)

bad_labels = sorted(
    observed_labels
    - allowed_labels
)

bad_splits = sorted(
    observed_splits
    - allowed_splits
)

if bad_labels:
    raise ValueError(
        f"Unexpected labels: {bad_labels}"
    )

if bad_splits:
    raise ValueError(
        f"Unexpected splits: {bad_splits}"
    )

# ------------------------------------------------------------
# Data accounting equality: every input must be SUCCESS/OK,
# SKIPPED, or ERROR. This mirrors the team SSOT accounting gate.
# ------------------------------------------------------------
success_count = int(
    metadata["status"]
    .isin(accepted_statuses)
    .sum()
)

skipped_count = int(
    metadata["status"]
    .eq("skipped")
    .sum()
)

error_count = int(
    metadata["status"]
    .eq("error")
    .sum()
)

if len(metadata) != (
    success_count
    + skipped_count
    + error_count
):
    raise RuntimeError(
        "Data accounting equality failed: "
        f"total={len(metadata)} success={success_count} "
        f"skipped={skipped_count} error={error_count}"
    )

accepted_mask = metadata["status"].isin(
    accepted_statuses
)

audit_only = (
    metadata.loc[
        ~accepted_mask
    ]
    .copy()
)

accepted = (
    metadata.loc[
        accepted_mask
    ]
    .copy()
)

if accepted.empty:
    raise RuntimeError(
        "No accepted eyebrow ROI rows exist."
    )

# Only accepted rows are required to have sample_id.
missing_sample_mask = (
    accepted["sample_id"].isna()
    | accepted["sample_id"].eq("")
)

if missing_sample_mask.any():
    bad = accepted.loc[
        missing_sample_mask,
        [
            "label",
            "split",
            "status",
        ],
    ].head(20)

    raise ValueError(
        "Accepted eyebrow ROI rows with missing sample_id:\n"
        f"{bad}"
    )

duplicate_sample_mask = (
    accepted["sample_id"]
    .duplicated(
        keep=False
    )
)

if duplicate_sample_mask.any():
    examples = (
        accepted.loc[
            duplicate_sample_mask,
            "sample_id",
        ]
        .head(20)
        .tolist()
    )

    raise ValueError(
        "Duplicate sample_id values among accepted rows. "
        f"Examples: {examples}"
    )


def _clean_optional_text(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    if not value:
        return None

    return value


def resolve_eyebrow_path(row) -> Path:
    # Prefer relative paths so the notebook stays portable across Drive mounts.
    relative_value = (
        _clean_optional_text(
            row.get(
                "output_relative_path"
            )
        )
        if "output_relative_path" in metadata.columns
        else None
    )

    absolute_value = (
        _clean_optional_text(
            row.get(
                "output_path"
            )
        )
        if "output_path" in metadata.columns
        else None
    )

    if relative_value is not None:
        return KAS_ROOT / relative_value

    if absolute_value is None:
        raise ValueError(
            f"No output path for sample_id={row.get('sample_id')}"
        )

    path = Path(
        absolute_value
    )

    if path.is_absolute():
        return path

    return KAS_ROOT / path


accepted[
    "resolved_eyebrow_path"
] = accepted.apply(
    resolve_eyebrow_path,
    axis=1,
)

accepted[
    "path_exists"
] = (
    accepted[
        "resolved_eyebrow_path"
    ]
    .map(
        lambda path: path.is_file()
    )
)

missing_file_rows = (
    accepted.loc[
        ~accepted["path_exists"]
    ]
    .copy()
)

eligible = (
    accepted.loc[
        accepted["path_exists"]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

if eligible.empty:
    raise RuntimeError(
        "No eligible eyebrow ROI images were found."
    )

# ------------------------------------------------------------
# Split + class quality gates
# ------------------------------------------------------------
split_class_table = pd.crosstab(
    eligible["split"],
    eligible["label"],
)

for split in [
    "train",
    "val",
    "test",
]:
    split_rows = (
        eligible.loc[
            eligible["split"]
            == split
        ]
    )

    if split_rows.empty:
        raise ValueError(
            f"Required split is absent: {split}"
        )

    labels_here = set(
        split_rows["label"]
        .dropna()
        .unique()
    )

    if labels_here != allowed_labels:
        raise ValueError(
            f"Split {split!r} must contain both classes. "
            f"Found: {sorted(labels_here)}"
        )

# Exact path leakage check.
split_paths = {
    split: set(
        eligible.loc[
            eligible["split"] == split,
            "resolved_eyebrow_path",
        ].astype(str)
    )
    for split in [
        "train",
        "val",
        "test",
    ]
}

path_intersections = {
    "train_val": len(
        split_paths["train"]
        & split_paths["val"]
    ),
    "train_test": len(
        split_paths["train"]
        & split_paths["test"]
    ),
    "val_test": len(
        split_paths["val"]
        & split_paths["test"]
    ),
}

if any(
    path_intersections.values()
):
    raise ValueError(
        "Cross-split duplicate eyebrow ROI paths detected: "
        f"{path_intersections}"
    )

# Exact source-frame hash leakage check.
hash_intersections = {}
if "input_sha256" in eligible.columns:
    hash_sets = {
        split: set(
            eligible.loc[
                eligible["split"] == split,
                "input_sha256",
            ]
            .dropna()
            .loc[lambda s: s.ne("")]
            .astype(str)
        )
        for split in [
            "train",
            "val",
            "test",
        ]
    }

    hash_intersections = {
        "train_val": len(
            hash_sets["train"]
            & hash_sets["val"]
        ),
        "train_test": len(
            hash_sets["train"]
            & hash_sets["test"]
        ),
        "val_test": len(
            hash_sets["val"]
            & hash_sets["test"]
        ),
    }

    if any(
        hash_intersections.values()
    ):
        raise ValueError(
            "Cross-split duplicate source-frame hashes detected: "
            f"{hash_intersections}"
        )

# ------------------------------------------------------------
# True source-video / person isolation check.
# Never fabricate source identity from filenames.
# ------------------------------------------------------------
source_group_status = "NOT_VERIFIABLE"
source_group_column = None
source_group_intersections = {}

for candidate in [
    "source_video",
    "person_id",
    "subject_id",
]:
    if candidate not in eligible.columns:
        continue

    values = (
        eligible[candidate]
        .astype("string")
        .str.strip()
    )

    populated = (
        values.notna()
        & values.ne("")
    )

    if not populated.any():
        continue

    if not populated.all():
        raise ValueError(
            f"{candidate} is only partially populated. "
            "A trustworthy group leakage check requires all eligible rows."
        )

    source_group_column = candidate

    split_groups = {
        split: set(
            values.loc[
                eligible["split"]
                .eq(split)
            ]
            .astype(str)
        )
        for split in [
            "train",
            "val",
            "test",
        ]
    }

    source_group_intersections = {
        "train_val": len(
            split_groups["train"]
            & split_groups["val"]
        ),
        "train_test": len(
            split_groups["train"]
            & split_groups["test"]
        ),
        "val_test": len(
            split_groups["val"]
            & split_groups["test"]
        ),
    }

    if any(
        source_group_intersections.values()
    ):
        raise ValueError(
            f"Cross-split {candidate} leakage detected: "
            f"{source_group_intersections}"
        )

    source_group_status = "VERIFIED"
    break

if source_group_status != "VERIFIED":
    warning = (
        "WARNING: True video/person-level split isolation cannot be "
        "verified because source_video/person_id is not populated. "
        "The notebook will preserve the upstream train/val/test split "
        "and has verified exact path/hash isolation only."
    )

    print("\n" + "!" * 80)
    print(warning)
    print("!" * 80 + "\n")
    LOGGER.warning(warning)

    if not CONFIG.allow_unverifiable_source_split:
        raise RuntimeError(
            "Strict source-group split quality gate blocked training."
        )

# ------------------------------------------------------------
# Standardized training metadata snapshot
# ------------------------------------------------------------
standardized = pd.DataFrame({
    "sample_id": eligible["sample_id"].astype(str),
    "source_video": (
        eligible["source_video"]
        if "source_video" in eligible.columns
        else pd.NA
    ),
    "frame_index": (
        eligible["frame_index"]
        if "frame_index" in eligible.columns
        else pd.NA
    ),
    "face_index": (
        eligible["face_index"]
        if "face_index" in eligible.columns
        else 0
    ),
    "roi_state": "eyebrow_roi",
    "label": eligible["label"],
    "split": eligible["split"],
    "status": "SUCCESS",
    "skip_reason": "",
    "sha256": (
        eligible["output_sha256"]
        if "output_sha256" in eligible.columns
        else pd.NA
    ),
    "output_path": (
        eligible["resolved_eyebrow_path"]
        .astype(str)
    ),
    "run_id": RUN_ID,
})

# Assert uniqueness after standardization.
if not standardized["sample_id"].is_unique:
    raise RuntimeError(
        "Standardized metadata has duplicate sample_id."
    )

atomic_csv_dump(
    standardized,
    DIRS["artifacts"]
    / "training_metadata_standardized.csv",
)

status_counts = (
    metadata["status"]
    .fillna("<missing>")
    .value_counts(
        dropna=False
    )
)

data_accounting = {
    "run_id": RUN_ID,
    "metadata_path": str(
        METADATA_PATH
    ),
    "total_metadata_rows": int(
        len(metadata)
    ),
    "success_status_rows": (
        success_count
    ),
    "skipped_status_rows": (
        skipped_count
    ),
    "error_status_rows": (
        error_count
    ),
    "audit_only_rows": int(
        len(audit_only)
    ),
    "missing_files_among_success": int(
        len(missing_file_rows)
    ),
    "training_eligible_success_count": int(
        len(eligible)
    ),
    "status_counts": {
        str(key): int(value)
        for key, value
        in status_counts.items()
    },
    "split_class_counts": {
        split: {
            label: int(
                (
                    eligible["split"].eq(split)
                    & eligible["label"].eq(label)
                ).sum()
            )
            for label in sorted(
                allowed_labels
            )
        }
        for split in [
            "train",
            "val",
            "test",
        ]
    },
    "cross_split_path_intersections": (
        path_intersections
    ),
    "cross_split_input_sha256_intersections": (
        hash_intersections
    ),
    "source_group_column": (
        source_group_column
    ),
    "source_group_leakage_status": (
        source_group_status
    ),
    "source_group_intersections": (
        source_group_intersections
    ),
    "source_group_note": (
        "VERIFIED means a populated source_video/person/subject ID "
        "was checked with disjoint sets. NOT_VERIFIABLE means the "
        "upstream metadata did not provide authoritative source identity."
    ),
}

atomic_json_dump(
    data_accounting,
    RUN_DIR
    / "data_accounting.json",
)

atomic_csv_dump(
    eligible,
    DIRS["artifacts"]
    / "eligible_metadata_before_cache.csv",
)

atomic_csv_dump(
    audit_only,
    DIRS["artifacts"]
    / "skipped_or_error_metadata.csv",
)

LOGGER.info(
    "Metadata validated: total=%d eligible=%d skipped=%d error=%d",
    len(metadata),
    len(eligible),
    skipped_count,
    error_count,
)

print("=" * 80)
print("EYEBROW METADATA VALIDATION PASSED")
print("=" * 80)
print(status_counts)
print()
print(split_class_table)
print()
print(
    json.dumps(
        data_accounting,
        indent=2,
        ensure_ascii=False,
    )
)



!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

EYEBROW METADATA VALIDATION PASSED
status
success    1962
skipped    1038
Name: count, dtype: Int64

label  fake  real
split            
test     95   101
train   776   786
val      98   106

{
  "run_id": "20260808_1248_eyebrow_efficientnet_b0_seed42",
  "metadata_path": "/content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Kaş/metadata.csv",
  "total_metadata_rows": 3000,
  "success_status_rows": 1962,
  "skipped_status_rows": 1038,
  "error_status_rows": 0,
  "audit_only_rows": 1038,
  "missing_files_among_success": 0,
  "training_eligible_success_count": 1962,
  "status_counts": {
    "success": 1962,
    "skipped": 1038
  },
  "split_class_counts": {
    "train": {
      "fake": 776,
      "real": 786
    },
    "val": {
      "fake": 98,
      "real": 106
    },
    "test": {
      "fake": 95,
   

In [9]:
# ============================================================
# CELL 9 — LOCAL SSD EYEBROW IMAGE CACHE
# ============================================================

LOCAL_CACHE_ROOT = Path(
    "/content/eyebrow_roi_cache"
)

LOCAL_CACHE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

def verify_image_file(path: Path) -> None:
    with Image.open(path) as image:
        image.verify()


def cache_one_image(
    source: Path,
    sample_id: str,
) -> Path:
    suffix = (
        source.suffix.lower()
        if source.suffix
        else ".jpg"
    )

    destination = (
        LOCAL_CACHE_ROOT
        / f"{sample_id}{suffix}"
    )

    if destination.is_file():
        try:
            if destination.stat().st_size > 0:
                if CONFIG.verify_cached_images:
                    verify_image_file(destination)

                return destination
        except Exception:
            try:
                destination.unlink()
            except FileNotFoundError:
                pass

    tmp = destination.with_suffix(
        destination.suffix
        + ".tmp"
    )

    shutil.copyfile(
        source,
        tmp,
    )

    if tmp.stat().st_size <= 0:
        raise RuntimeError(
            f"Copied cache file is empty: {source}"
        )

    if CONFIG.verify_cached_images:
        verify_image_file(tmp)

    os.replace(
        tmp,
        destination,
    )

    return destination


if CONFIG.cache_images_locally:
    cached_paths = []

    iterator = tqdm(
        eligible.itertuples(
            index=False
        ),
        total=len(eligible),
        desc="Caching eyebrow ROI images to local SSD",
    )

    for row in iterator:
        source = Path(
            row.resolved_eyebrow_path
        )

        cached = cache_one_image(
            source=source,
            sample_id=str(
                row.sample_id
            ),
        )

        cached_paths.append(
            str(cached)
        )

    eligible[
        "training_eyebrow_path"
    ] = cached_paths
else:
    eligible[
        "training_eyebrow_path"
    ] = (
        eligible[
            "resolved_eyebrow_path"
        ].astype(str)
    )

# Final cache accounting.
if eligible[
    "training_eyebrow_path"
].isna().any():
    raise RuntimeError(
        "Missing training_eyebrow_path after cache step."
    )

missing_cached = [
    path
    for path in eligible[
        "training_eyebrow_path"
    ]
    if not Path(path).is_file()
]

if missing_cached:
    raise RuntimeError(
        "Local cache validation failed. "
        f"Missing files: {len(missing_cached)}"
    )

atomic_csv_dump(
    eligible,
    DIRS["artifacts"]
    / "eligible_metadata.csv",
)

LOGGER.info(
    "Local cache ready: %d images at %s",
    len(eligible),
    LOCAL_CACHE_ROOT,
)

print("=" * 80)
print("LOCAL EYEBROW CACHE PASSED")
print("=" * 80)
print("Cached/usable images:", len(eligible))
print("Local cache root     :", LOCAL_CACHE_ROOT)


Caching eyebrow ROI images to local SSD:   0%|          | 0/1962 [00:00<?, ?it/s]

LOCAL EYEBROW CACHE PASSED
Cached/usable images: 1962
Local cache root     : /content/eyebrow_roi_cache


In [10]:
# ============================================================
# CELL 10 — EYEBROW-AWARE TRANSFORMS + DATASETS + DATALOADERS
# ============================================================

IMAGENET_MEAN = [
    0.485,
    0.456,
    0.406,
]

IMAGENET_STD = [
    0.229,
    0.224,
    0.225,
]


class ResizePadSquare:
    """
    Preserve the eyebrow ROI aspect ratio, fit it inside a square canvas,
    and pad the remaining area with the ImageNet mean RGB value.

    This avoids vertically stretching a wide, short eyebrow crop into a
    square before EfficientNet-B0.
    """
    def __init__(
        self,
        size: int,
    ):
        self.size = int(
            size
        )

        self.fill = tuple(
            int(round(value * 255))
            for value in IMAGENET_MEAN
        )

    def __call__(
        self,
        image: Image.Image,
    ) -> Image.Image:
        image = image.convert(
            "RGB"
        )

        width, height = (
            image.size
        )

        if width <= 0 or height <= 0:
            raise ValueError(
                f"Invalid image size: {image.size}"
            )

        scale = min(
            self.size / width,
            self.size / height,
        )

        new_width = max(
            1,
            int(round(width * scale)),
        )

        new_height = max(
            1,
            int(round(height * scale)),
        )

        resized = image.resize(
            (
                new_width,
                new_height,
            ),
            resample=Image.Resampling.BICUBIC,
        )

        canvas = Image.new(
            "RGB",
            (
                self.size,
                self.size,
            ),
            self.fill,
        )

        left = (
            self.size
            - new_width
        ) // 2

        top = (
            self.size
            - new_height
        ) // 2

        canvas.paste(
            resized,
            (
                left,
                top,
            ),
        )

        return canvas


resize_pad = ResizePadSquare(
    CONFIG.image_size
)

# Augmentation is TRAIN ONLY.
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(
        p=0.5
    ),
    transforms.ColorJitter(
        brightness=0.10,
        contrast=0.10,
    ),
    transforms.Lambda(
        resize_pad
    ),
    transforms.ToTensor(),
    # These are the prescribed pretrained ImageNet statistics,
    # not statistics estimated from validation/test data.
    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD,
    ),
])

eval_transform = transforms.Compose([
    transforms.Lambda(
        resize_pad
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        IMAGENET_MEAN,
        IMAGENET_STD,
    ),
])

LABEL_TO_INT = {
    CONFIG.negative_label: 0,
    CONFIG.positive_label: 1,
}

INT_TO_LABEL = {
    0: CONFIG.negative_label,
    1: CONFIG.positive_label,
}


class EyebrowROIDataset(Dataset):
    def __init__(
        self,
        frame: pd.DataFrame,
        transform,
    ):
        self.df = (
            frame
            .reset_index(
                drop=True
            )
            .copy()
        )

        self.transform = (
            transform
        )

    def __len__(
        self,
    ):
        return len(
            self.df
        )

    def __getitem__(
        self,
        index,
    ):
        row = self.df.iloc[
            index
        ]

        path = Path(
            row[
                "training_eyebrow_path"
            ]
        )

        try:
            with Image.open(
                path
            ) as image:
                image = (
                    image
                    .convert("RGB")
                )

                tensor = (
                    self.transform(
                        image
                    )
                )
        except Exception as e:
            raise RuntimeError(
                "Failed to read/transform eyebrow ROI. "
                f"index={index}, path={path}"
            ) from e

        label = torch.tensor(
            LABEL_TO_INT[
                row["label"]
            ],
            dtype=torch.float32,
        )

        return {
            "image": tensor,
            "label": label,
            "sample_id": str(
                row["sample_id"]
            ),
            "path": str(path),
        }


split_frames = {
    split: (
        eligible.loc[
            eligible["split"]
            == split
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )
    for split in [
        "train",
        "val",
        "test",
    ]
}

datasets = {
    "train": EyebrowROIDataset(
        split_frames[
            "train"
        ],
        train_transform,
    ),
    "val": EyebrowROIDataset(
        split_frames[
            "val"
        ],
        eval_transform,
    ),
    "test": EyebrowROIDataset(
        split_frames[
            "test"
        ],
        eval_transform,
    ),
}

loader_generator = torch.Generator()
loader_generator.manual_seed(
    CONFIG.seed
)

loaders = {
    "train": DataLoader(
        datasets["train"],
        batch_size=(
            CONFIG.batch_size
        ),
        shuffle=True,
        num_workers=(
            CONFIG.num_workers
        ),
        pin_memory=(
            DEVICE.type
            == "cuda"
        ),
        generator=(
            loader_generator
        ),
        drop_last=False,
    ),
    "val": DataLoader(
        datasets["val"],
        batch_size=(
            CONFIG.batch_size
        ),
        shuffle=False,
        num_workers=(
            CONFIG.num_workers
        ),
        pin_memory=(
            DEVICE.type
            == "cuda"
        ),
        drop_last=False,
    ),
    "test": DataLoader(
        datasets["test"],
        batch_size=(
            CONFIG.batch_size
        ),
        shuffle=False,
        num_workers=(
            CONFIG.num_workers
        ),
        pin_memory=(
            DEVICE.type
            == "cuda"
        ),
        drop_last=False,
    ),
}

print({
    split: len(
        loader.dataset
    )
    for split, loader
    in loaders.items()
})

sample_batch = next(
    iter(
        loaders["train"]
    )
)

expected_shape = (
    3,
    CONFIG.image_size,
    CONFIG.image_size,
)

if tuple(
    sample_batch[
        "image"
    ].shape[1:]
) != expected_shape:
    raise RuntimeError(
        "Unexpected batch image shape: "
        f"{tuple(sample_batch['image'].shape)}"
    )

print(
    "Batch shape:",
    tuple(
        sample_batch[
            "image"
        ].shape
    ),
)


{'train': 1562, 'val': 204, 'test': 196}
Batch shape: (32, 3, 224, 224)


In [12]:

# ============================================================
# CELL 11 — 5-BATCH I/O SPEED SANITY CHECK
# ============================================================

speed_test_batches = 5

start = time.time()
seen = 0

for batch_index, batch in enumerate(
    tqdm(
        loaders["train"],
        total=min(
            speed_test_batches,
            len(loaders["train"]),
        ),
        desc="DataLoader speed test",
    )
):
    _ = batch["image"]

    seen += 1

    if seen >= speed_test_batches:
        break

elapsed = time.time() - start

images_seen = min(
    seen * CONFIG.batch_size,
    len(loaders["train"].dataset),
)

print(
    f"{seen} batches / ~{images_seen} images "
    f"loaded in {elapsed:.2f} seconds."
)

if elapsed > 60:
    print(
        "WARNING: Local data loading is unusually slow. "
        "Training will still run, but inspect Colab disk/runtime health."
    )
else:
    print(
        "DataLoader speed looks healthy."
    )


DataLoader speed test:   0%|          | 0/5 [00:00<?, ?it/s]

5 batches / ~160 images loaded in 0.37 seconds.
DataLoader speed looks healthy.


In [13]:
# ============================================================
# CELL 12 — MODEL
# ============================================================

def build_model(
    pretrained: bool = True,
) -> nn.Module:
    weights = (
        EfficientNet_B0_Weights.DEFAULT
        if pretrained
        else None
    )

    try:
        model = efficientnet_b0(
            weights=weights
        )
    except Exception as e:
        raise RuntimeError(
            "EfficientNet-B0 pretrained weights could not be loaded. "
            "Check Colab internet/runtime availability."
        ) from e

    in_features = (
        model
        .classifier[1]
        .in_features
    )

    model.classifier = nn.Sequential(
        nn.Dropout(
            p=CONFIG.dropout
        ),
        nn.Linear(
            in_features,
            1,
        ),
    )

    return model


model = build_model(
    pretrained=True
).to(DEVICE)

total_parameters = sum(
    parameter.numel()
    for parameter
    in model.parameters()
)

print(
    "Total parameters:",
    f"{total_parameters:,}",
)

atomic_write_text(
    str(model),
    DIRS["artifacts"]
    / "model_architecture.txt",
)

LOGGER.info(
    "EfficientNet-B0 model built with %d parameters",
    total_parameters,
)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 164MB/s]


Total parameters: 4,008,829


In [14]:

# ============================================================
# CELL 13 — METRICS
# ============================================================

def safe_roc_auc(
    y_true,
    probabilities,
):
    if len(
        np.unique(
            y_true
        )
    ) < 2:
        return float("nan")

    return float(
        roc_auc_score(
            y_true,
            probabilities,
        )
    )


def safe_average_precision(
    y_true,
    probabilities,
):
    if len(
        np.unique(
            y_true
        )
    ) < 2:
        return float("nan")

    return float(
        average_precision_score(
            y_true,
            probabilities,
        )
    )


def binary_metrics(
    y_true,
    probabilities,
    threshold: float,
) -> Dict[str, float]:
    y_true = np.asarray(
        y_true,
        dtype=int,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    predictions = (
        probabilities
        >= threshold
    ).astype(int)

    tn, fp, fn, tp = (
        confusion_matrix(
            y_true,
            predictions,
            labels=[
                0,
                1,
            ],
        )
        .ravel()
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else float("nan")
    )

    return {
        "threshold": float(
            threshold
        ),
        "accuracy": float(
            accuracy_score(
                y_true,
                predictions,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                predictions,
            )
        ),
        "precision": float(
            precision_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "recall": float(
            recall_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "f1": float(
            f1_score(
                y_true,
                predictions,
                zero_division=0,
            )
        ),
        "specificity": float(
            specificity
        ),
        "roc_auc": safe_roc_auc(
            y_true,
            probabilities,
        ),
        "average_precision": (
            safe_average_precision(
                y_true,
                probabilities,
            )
        ),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def select_validation_threshold(
    y_true,
    probabilities,
):
    thresholds = np.linspace(
        CONFIG.threshold_min,
        CONFIG.threshold_max,
        CONFIG.threshold_steps,
    )

    rows = [
        binary_metrics(
            y_true,
            probabilities,
            float(threshold),
        )
        for threshold
        in thresholds
    ]

    table = pd.DataFrame(
        rows
    )

    table[
        "distance_to_0_5"
    ] = (
        table["threshold"]
        - 0.5
    ).abs()

    best = (
        table
        .sort_values(
            [
                "f1",
                "balanced_accuracy",
                "distance_to_0_5",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .iloc[0]
    )

    return (
        float(
            best["threshold"]
        ),
        table.drop(
            columns=[
                "distance_to_0_5"
            ]
        ),
    )


In [15]:
# ============================================================
# CELL 14 — SAFE CHECKPOINT + RNG HELPERS
# ============================================================

CHECKPOINT_SCHEMA_VERSION = 2

def get_rng_state() -> dict:
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": (
            torch.get_rng_state()
            .cpu()
        ),
        "loader_generator": (
            loader_generator
            .get_state()
            .cpu()
        ),
    }

    if torch.cuda.is_available():
        state["cuda"] = [
            cuda_state.cpu()
            for cuda_state
            in torch.cuda.get_rng_state_all()
        ]

    return state


def _to_cpu_byte_tensor(
    value,
) -> torch.Tensor:
    if not torch.is_tensor(
        value
    ):
        value = torch.tensor(
            value,
            dtype=torch.uint8,
        )

    return (
        value
        .detach()
        .cpu()
        .to(
            dtype=torch.uint8
        )
    )


def set_rng_state(
    state: Optional[dict],
) -> None:
    if not state:
        return

    python_state = state.get(
        "python"
    )

    if python_state is not None:
        random.setstate(
            python_state
        )

    numpy_state = state.get(
        "numpy"
    )

    if numpy_state is not None:
        np.random.set_state(
            numpy_state
        )

    torch_state = state.get(
        "torch"
    )

    if torch_state is not None:
        torch.set_rng_state(
            _to_cpu_byte_tensor(
                torch_state
            )
        )

    loader_state = state.get(
        "loader_generator"
    )

    if loader_state is not None:
        loader_generator.set_state(
            _to_cpu_byte_tensor(
                loader_state
            )
        )

    cuda_states = state.get(
        "cuda"
    )

    if (
        torch.cuda.is_available()
        and cuda_states is not None
    ):
        safe_states = [
            _to_cpu_byte_tensor(
                cuda_state
            )
            for cuda_state
            in cuda_states
        ]

        if len(
            safe_states
        ) == torch.cuda.device_count():
            torch.cuda.set_rng_state_all(
                safe_states
            )
        else:
            print(
                "WARNING: CUDA RNG device count differs; "
                "CUDA RNG restore skipped."
            )


def save_checkpoint(
    path: Path,
    *,
    epoch: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scheduler,
    best_score: float,
    history: List[dict],
    stage_name: str,
) -> None:
    state = {
        "checkpoint_schema_version": (
            CHECKPOINT_SCHEMA_VERSION
        ),
        "run_schema_version": (
            RUN_SCHEMA_VERSION
        ),
        "epoch": int(
            epoch
        ),
        "stage_name": (
            stage_name
        ),
        "model_state_dict": (
            model.state_dict()
        ),
        "optimizer_state_dict": (
            optimizer.state_dict()
        ),
        "scheduler_state_dict": (
            scheduler.state_dict()
            if scheduler is not None
            else None
        ),
        # FP32 notebook: no GradScaler is used, but the field is explicit
        # for checkpoint schema completeness.
        "scaler_state_dict": None,
        "best_metric_score": float(
            best_score
        ),
        "history": history,
        "config": asdict(
            CONFIG
        ),
        "rng_state": get_rng_state(),
    }

    atomic_torch_save(
        state,
        path,
    )


def checkpoint_is_compatible(
    path: Path,
    stage_name: str,
) -> bool:
    if not path.is_file():
        return False

    try:
        checkpoint = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

        return (
            int(
                checkpoint.get(
                    "checkpoint_schema_version",
                    -1,
                )
            )
            == CHECKPOINT_SCHEMA_VERSION
            and int(
                checkpoint.get(
                    "run_schema_version",
                    -1,
                )
            )
            == RUN_SCHEMA_VERSION
            and checkpoint.get(
                "stage_name"
            )
            == stage_name
        )
    except Exception:
        return False


def load_checkpoint(
    path: Path,
    *,
    stage_name: str,
    model: nn.Module,
    optimizer: Optional[
        torch.optim.Optimizer
    ] = None,
    scheduler=None,
) -> dict:
    if not checkpoint_is_compatible(
        path,
        stage_name,
    ):
        raise RuntimeError(
            "Checkpoint is missing, corrupt, or incompatible:\n"
            f"{path}"
        )

    # Always load checkpoint on CPU first.
    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )

    model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    model.to(
        DEVICE
    )

    if optimizer is not None:
        optimizer.load_state_dict(
            checkpoint[
                "optimizer_state_dict"
            ]
        )

        # Move optimizer tensors to current device.
        for optimizer_state in (
            optimizer.state.values()
        ):
            for key, value in list(
                optimizer_state.items()
            ):
                if torch.is_tensor(
                    value
                ):
                    optimizer_state[
                        key
                    ] = value.to(
                        DEVICE
                    )

    if (
        scheduler is not None
        and checkpoint.get(
            "scheduler_state_dict"
        ) is not None
    ):
        scheduler.load_state_dict(
            checkpoint[
                "scheduler_state_dict"
            ]
        )

    set_rng_state(
        checkpoint.get(
            "rng_state"
        )
    )

    return checkpoint


In [16]:

# ============================================================
# CELL 15 — SAFE FP32 EPOCH RUNNER WITH LIVE PROGRESS
# ============================================================

criterion = (
    nn.BCEWithLogitsLoss()
)


def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    *,
    training: bool,
    optimizer: Optional[
        torch.optim.Optimizer
    ] = None,
    frozen_backbone: bool = False,
    description: str = "",
) -> dict:
    if training:
        if optimizer is None:
            raise ValueError(
                "optimizer is required when training=True"
            )

        model.train()

        # True frozen backbone: keep BatchNorm/dropout in features frozen too.
        if frozen_backbone:
            model.features.eval()
            model.classifier.train()
    else:
        model.eval()

    total_loss = 0.0
    processed = 0

    all_labels = []
    all_probabilities = []
    all_sample_ids = []
    all_paths = []

    progress = tqdm(
        loader,
        total=len(loader),
        desc=description,
        leave=False,
    )

    for batch_index, batch in enumerate(
        progress
    ):
        images = batch[
            "image"
        ].to(
            DEVICE,
            non_blocking=True,
        )

        labels = batch[
            "label"
        ].to(
            DEVICE,
            non_blocking=True,
        ).float()

        if not torch.isfinite(
            images
        ).all():
            raise FloatingPointError(
                f"NaN/Inf detected in images at batch={batch_index}"
            )

        if not torch.isfinite(
            labels
        ).all():
            raise FloatingPointError(
                f"NaN/Inf detected in labels at batch={batch_index}"
            )

        if training:
            optimizer.zero_grad(
                set_to_none=True
            )

        with torch.set_grad_enabled(
            training
        ):
            # Deliberately FP32 for numerical stability.
            logits = (
                model(images)
                .squeeze(1)
                .float()
            )

            if not torch.isfinite(
                logits
            ).all():
                raise FloatingPointError(
                    f"Non-finite logits at batch={batch_index}"
                )

            loss = criterion(
                logits,
                labels,
            )

            if not torch.isfinite(
                loss
            ):
                raise FloatingPointError(
                    f"Non-finite loss at batch={batch_index}"
                )

            if training:
                loss.backward()

                bad_gradient_names = []

                for (
                    parameter_name,
                    parameter,
                ) in model.named_parameters():
                    if parameter.grad is None:
                        continue

                    if not torch.isfinite(
                        parameter.grad
                    ).all():
                        bad_gradient_names.append(
                            parameter_name
                        )

                if bad_gradient_names:
                    raise FloatingPointError(
                        "Non-finite gradients detected at "
                        f"batch={batch_index}. "
                        "First affected parameters: "
                        f"{bad_gradient_names[:10]}"
                    )

                gradient_norm = (
                    torch.nn.utils
                    .clip_grad_norm_(
                        model.parameters(),
                        CONFIG.grad_clip_norm,
                        error_if_nonfinite=True,
                    )
                )

                if not torch.isfinite(
                    torch.as_tensor(
                        gradient_norm
                    )
                ):
                    raise FloatingPointError(
                        f"Non-finite gradient norm at batch={batch_index}"
                    )

                optimizer.step()

        probabilities = torch.sigmoid(
            logits.detach()
        )

        if not torch.isfinite(
            probabilities
        ).all():
            raise FloatingPointError(
                f"Non-finite probabilities at batch={batch_index}"
            )

        batch_size = images.size(
            0
        )

        total_loss += (
            float(
                loss.detach().item()
            )
            * batch_size
        )

        processed += (
            batch_size
        )

        all_labels.extend(
            labels
            .detach()
            .cpu()
            .numpy()
            .astype(int)
            .tolist()
        )

        all_probabilities.extend(
            probabilities
            .detach()
            .cpu()
            .numpy()
            .astype(float)
            .tolist()
        )

        all_sample_ids.extend(
            list(
                batch[
                    "sample_id"
                ]
            )
        )

        all_paths.extend(
            list(
                batch[
                    "path"
                ]
            )
        )

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            processed=processed,
        )

    if processed != len(
        loader.dataset
    ):
        raise RuntimeError(
            "Epoch accounting mismatch: "
            f"processed={processed}, dataset={len(loader.dataset)}"
        )

    return {
        "loss": (
            total_loss
            / max(
                processed,
                1,
            )
        ),
        "labels": np.asarray(
            all_labels,
            dtype=int,
        ),
        "probabilities": np.asarray(
            all_probabilities,
            dtype=float,
        ),
        "sample_ids": (
            all_sample_ids
        ),
        "paths": (
            all_paths
        ),
        "count": int(
            processed
        ),
    }


In [17]:

# ============================================================
# CELL 16 — TWO-BATCH MODEL SMOKE TEST
# ============================================================

def model_smoke_test(
    model: nn.Module,
) -> None:
    print(
        "Running two-batch forward/backward smoke test..."
    )

    backup_state = {
        key: value
        .detach()
        .cpu()
        .clone()
        for key, value
        in model.state_dict().items()
    }

    temporary_optimizer = (
        torch.optim.AdamW(
            model.parameters(),
            lr=1e-6,
        )
    )

    model.train()

    seen_batches = 0

    for batch in tqdm(
        loaders["train"],
        total=min(
            2,
            len(
                loaders["train"]
            ),
        ),
        desc="Smoke test",
        leave=False,
    ):
        images = (
            batch["image"]
            .to(DEVICE)
        )

        labels = (
            batch["label"]
            .to(DEVICE)
            .float()
        )

        temporary_optimizer.zero_grad(
            set_to_none=True
        )

        logits = (
            model(images)
            .squeeze(1)
            .float()
        )

        loss = criterion(
            logits,
            labels,
        )

        if not torch.isfinite(
            loss
        ):
            raise FloatingPointError(
                "Smoke test produced non-finite loss."
            )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            CONFIG.grad_clip_norm,
            error_if_nonfinite=True,
        )

        temporary_optimizer.step()

        seen_batches += 1

        if seen_batches >= 2:
            break

    if seen_batches < 2:
        raise RuntimeError(
            "Smoke test could not obtain two batches."
        )

    model.load_state_dict(
        backup_state
    )

    model.to(
        DEVICE
    )

    del (
        backup_state,
        temporary_optimizer,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(
        "Smoke test PASSED."
    )


model_smoke_test(
    model
)


# ------------------------------------------------------------
# CHECKPOINT ROUND-TRIP QUALITY GATE
# ------------------------------------------------------------

def checkpoint_roundtrip_test(
    model: nn.Module,
) -> None:
    print(
        "Running checkpoint round-trip quality gate..."
    )

    stage_name = (
        "checkpoint_smoke"
    )

    temp_path = (
        DIRS["checkpoints"]
        / "_checkpoint_smoke.ckpt"
    )

    optimizer = (
        torch.optim.AdamW(
            model.parameters(),
            lr=1e-6,
        )
    )

    batch = next(
        iter(
            loaders["train"]
        )
    )

    images = (
        batch["image"][
            : min(
                4,
                len(
                    batch["image"]
                ),
            )
        ]
        .to(DEVICE)
    )

    labels = (
        batch["label"][
            : len(images)
        ]
        .to(DEVICE)
        .float()
    )

    model.eval()

    with torch.no_grad():
        logits_before = (
            model(images)
            .squeeze(1)
            .float()
        )

        loss_before = (
            criterion(
                logits_before,
                labels,
            )
            .detach()
            .cpu()
        )

    save_checkpoint(
        temp_path,
        epoch=0,
        model=model,
        optimizer=optimizer,
        scheduler=None,
        best_score=0.0,
        history=[],
        stage_name=stage_name,
    )

    reloaded_model = (
        build_model(
            pretrained=False
        )
        .to(DEVICE)
    )

    reloaded_optimizer = (
        torch.optim.AdamW(
            reloaded_model.parameters(),
            lr=1e-6,
        )
    )

    load_checkpoint(
        temp_path,
        stage_name=stage_name,
        model=reloaded_model,
        optimizer=reloaded_optimizer,
        scheduler=None,
    )

    reloaded_model.eval()

    with torch.no_grad():
        logits_after = (
            reloaded_model(images)
            .squeeze(1)
            .float()
        )

        loss_after = (
            criterion(
                logits_after,
                labels,
            )
            .detach()
            .cpu()
        )

    if not torch.allclose(
        loss_before,
        loss_after,
        atol=1e-7,
        rtol=1e-6,
    ):
        raise RuntimeError(
            "Checkpoint round-trip loss mismatch: "
            f"before={loss_before.item():.10f}, "
            f"after={loss_after.item():.10f}"
        )

    temp_path.unlink(
        missing_ok=True
    )

    del (
        optimizer,
        reloaded_model,
        reloaded_optimizer,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(
        "Checkpoint round-trip PASSED. "
        f"loss={loss_before.item():.8f}"
    )


checkpoint_roundtrip_test(
    model
)


Running two-batch forward/backward smoke test...


Smoke test:   0%|          | 0/2 [00:00<?, ?it/s]

Smoke test PASSED.
Running checkpoint round-trip quality gate...
Checkpoint round-trip PASSED. loss=0.65130931


In [18]:
# ============================================================
# CELL 17 — TRAINING FUNCTION
# ============================================================

def train_stage(
    *,
    model: nn.Module,
    stage_name: str,
    epochs: int,
    lr: float,
    stage_dir: Path,
    frozen_backbone: bool,
    allow_resume: bool,
) -> dict:
    trainable_parameters = [
        parameter
        for parameter
        in model.parameters()
        if parameter.requires_grad
    ]

    if not trainable_parameters:
        raise RuntimeError(
            f"No trainable parameters for stage={stage_name}"
        )

    optimizer = (
        torch.optim.AdamW(
            trainable_parameters,
            lr=lr,
            weight_decay=(
                CONFIG.weight_decay
            ),
        )
    )

    scheduler = (
        torch.optim.lr_scheduler
        .ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=(
                CONFIG.scheduler_patience
            ),
            min_lr=1e-7,
        )
    )

    last_checkpoint = (
        stage_dir
        / "last.ckpt"
    )

    best_checkpoint = (
        stage_dir
        / "best.ckpt"
    )

    history = []
    start_epoch = 1
    best_score = -float("inf")
    no_improvement_epochs = 0

    if (
        allow_resume
        and checkpoint_is_compatible(
            last_checkpoint,
            stage_name,
        )
    ):
        print(
            f"[{stage_name}] Resuming compatible checkpoint:"
        )

        print(
            last_checkpoint
        )

        checkpoint = load_checkpoint(
            last_checkpoint,
            stage_name=stage_name,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
        )

        history = list(
            checkpoint.get(
                "history",
                [],
            )
        )

        start_epoch = (
            int(
                checkpoint["epoch"]
            )
            + 1
        )

        best_score = float(
            checkpoint.get(
                "best_metric_score",
                -float("inf"),
            )
        )

        if history:
            monitor_scores = [
                row.get(
                    "val_roc_auc",
                    float("nan"),
                )
                for row
                in history
            ]

            safe_scores = [
                score
                if np.isfinite(score)
                else -float("inf")
                for score
                in monitor_scores
            ]

            best_index = int(
                np.argmax(
                    safe_scores
                )
            )

            no_improvement_epochs = max(
                0,
                len(history)
                - best_index
                - 1,
            )

    if start_epoch > epochs:
        print(
            f"[{stage_name}] Stage already complete."
        )

        if not checkpoint_is_compatible(
            best_checkpoint,
            stage_name,
        ):
            raise RuntimeError(
                f"Stage says complete but compatible best.ckpt is missing: "
                f"{best_checkpoint}"
            )

        return {
            "stage": stage_name,
            "history": history,
            "best_score": best_score,
            "best_checkpoint": str(
                best_checkpoint
            ),
            "last_checkpoint": str(
                last_checkpoint
            ),
        }

    for epoch in range(
        start_epoch,
        epochs + 1,
    ):
        epoch_start = time.time()

        train_output = run_epoch(
            model,
            loaders["train"],
            training=True,
            optimizer=optimizer,
            frozen_backbone=(
                frozen_backbone
            ),
            description=(
                f"{stage_name} train {epoch}/{epochs}"
            ),
        )

        validation_output = run_epoch(
            model,
            loaders["val"],
            training=False,
            frozen_backbone=False,
            description=(
                f"{stage_name} val {epoch}/{epochs}"
            ),
        )

        train_metrics = binary_metrics(
            train_output[
                "labels"
            ],
            train_output[
                "probabilities"
            ],
            threshold=0.5,
        )

        validation_metrics = (
            binary_metrics(
                validation_output[
                    "labels"
                ],
                validation_output[
                    "probabilities"
                ],
                threshold=0.5,
            )
        )

        monitor = (
            validation_metrics[
                "roc_auc"
            ]
        )

        if not np.isfinite(
            monitor
        ):
            monitor = (
                validation_metrics[
                    "f1"
                ]
            )

        scheduler.step(
            monitor
        )

        row = {
            "stage": stage_name,
            "epoch": int(
                epoch
            ),
            "lr": float(
                optimizer
                .param_groups[0][
                    "lr"
                ]
            ),
            "train_loss": float(
                train_output[
                    "loss"
                ]
            ),
            "val_loss": float(
                validation_output[
                    "loss"
                ]
            ),
            **{
                f"train_{key}": value
                for key, value
                in train_metrics.items()
            },
            **{
                f"val_{key}": value
                for key, value
                in validation_metrics.items()
            },
            "epoch_seconds": float(
                time.time()
                - epoch_start
            ),
        }

        history.append(
            row
        )

        improved = (
            monitor
            > best_score
            + 1e-8
        )

        if improved:
            best_score = float(
                monitor
            )
            no_improvement_epochs = 0
        else:
            no_improvement_epochs += 1

        # Every completed epoch is resumable.
        save_checkpoint(
            last_checkpoint,
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            best_score=best_score,
            history=history,
            stage_name=stage_name,
        )

        # Rotating epoch backups (keep the latest N).
        epoch_checkpoint = (
            stage_dir
            / f"epoch_{epoch:03d}.ckpt"
        )

        save_checkpoint(
            epoch_checkpoint,
            epoch=epoch,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            best_score=best_score,
            history=history,
            stage_name=stage_name,
        )

        epoch_backups = sorted(
            stage_dir.glob(
                "epoch_*.ckpt"
            )
        )

        while len(
            epoch_backups
        ) > CONFIG.epoch_checkpoint_keep:
            oldest = epoch_backups.pop(
                0
            )
            oldest.unlink()

        if improved:
            save_checkpoint(
                best_checkpoint,
                epoch=epoch,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                best_score=best_score,
                history=history,
                stage_name=stage_name,
            )

        atomic_csv_dump(
            pd.DataFrame(
                history
            ),
            DIRS["metrics"]
            / f"{stage_name}_training_history.csv",
        )

        message = (
            f"[{stage_name}] epoch {epoch:02d}/{epochs} | "
            f"train_loss={row['train_loss']:.5f} | "
            f"val_loss={row['val_loss']:.5f} | "
            f"val_auc={row['val_roc_auc']:.5f} | "
            f"val_f1={row['val_f1']:.5f} | "
            f"lr={row['lr']:.2e} | "
            f"time={row['epoch_seconds']:.1f}s"
        )

        print(
            message
        )

        LOGGER.info(
            message
        )

        if (
            no_improvement_epochs
            >= CONFIG.early_stopping_patience
        ):
            print(
                f"[{stage_name}] Early stopping after "
                f"{no_improvement_epochs} non-improving epoch(s)."
            )
            break

    if not checkpoint_is_compatible(
        best_checkpoint,
        stage_name,
    ):
        raise RuntimeError(
            f"No compatible best checkpoint created for stage={stage_name}"
        )

    return {
        "stage": stage_name,
        "history": history,
        "best_score": best_score,
        "best_checkpoint": str(
            best_checkpoint
        ),
        "last_checkpoint": str(
            last_checkpoint
        ),
    }


In [19]:

# ============================================================
# CELL 18 — STAGE 1: FROZEN EFFICIENTNET-B0 BACKBONE
# ============================================================

for parameter in (
    model.features.parameters()
):
    parameter.requires_grad = False

for parameter in (
    model.classifier.parameters()
):
    parameter.requires_grad = True

frozen_trainable = sum(
    parameter.numel()
    for parameter
    in model.parameters()
    if parameter.requires_grad
)

print(
    "Trainable params (frozen):",
    f"{frozen_trainable:,}",
)

frozen_summary = train_stage(
    model=model,
    stage_name="frozen",
    epochs=(
        CONFIG.frozen_epochs
    ),
    lr=(
        CONFIG.frozen_lr
    ),
    stage_dir=(
        FROZEN_DIR
    ),
    frozen_backbone=True,
    allow_resume=True,
)

atomic_json_dump(
    frozen_summary,
    DIRS["metrics"]
    / "frozen_training_summary.json",
)

frozen_summary


Trainable params (frozen): 1,281


frozen train 1/5:   0%|          | 0/49 [00:00<?, ?it/s]

frozen val 1/5:   0%|          | 0/7 [00:00<?, ?it/s]

[frozen] epoch 01/5 | train_loss=0.68310 | val_loss=0.67369 | val_auc=0.60281 | val_f1=0.55172 | lr=1.00e-03 | time=9.0s


frozen train 2/5:   0%|          | 0/49 [00:00<?, ?it/s]

...

[frozen] epoch 04/5 | train_loss=0.64427 | val_loss=0.66787 | val_auc=0.64247 | val_f1=0.39189 | lr=1.00e-03 | time=9.2s


frozen train 5/5:   0%|          | 0/49 [00:00<?, ?it/s]

frozen val 5/5:   0%|          | 0/7 [00:00<?, ?it/s]

[frozen] epoch 05/5 | train_loss=0.64022 | val_loss=0.67020 | val_auc=0.64411 | val_f1=0.63717 | lr=1.00e-03 | time=9.3s


{'stage': 'frozen',
 'history': [{'stage': 'frozen',
   'epoch': 1,
   'lr': 0.001,
   'train_loss': 0.6831013695378615,
   'val_loss': 0.6736945942336438,
   'train_threshold': 0.5,
   'train_accuracy': 0.5505761843790012,
   'train_balanced_accuracy': 0.5505413682747042,
   'train_precision': 0.5479274611398963,
   'train_recall': 0.5451030927835051,
   'train_f1': 0.5465116279069767,
   'train_specificity': 0.5559796437659033,
   'train_roc_auc': 0.5798190301933317,
   'train_average_precision': 0.5746767478497321,
   'train_tn': 437,
   'train_fp': 349,
   'train_fn': 353,
   'train_tp': 423,
   'val_threshold': 0.5,
   'val_accuracy': 0.553921568627451,
   'val_balanced_accuracy': 0.5545822102425876,
   'val_precision': 0.5333333333333333,
   'val_recall': 0.5714285714285714,
   'val_f1': 0.5517241379310345,
   'val_specificity': 0.5377358490566038,
   'val_roc_auc': 0.6028109356950327,
   'val_average_precision': 0.5885626711273588,
   'val_tn': 57,
   'val_fp': 49,
   'val_fn': 

In [20]:

# ============================================================
# CELL 19 — STAGE 2: FULL FP32 FINE-TUNING
# ============================================================

FROZEN_BEST = (
    FROZEN_DIR
    / "best.ckpt"
)

if not checkpoint_is_compatible(
    FROZEN_BEST,
    "frozen",
):
    raise RuntimeError(
        "Frozen best checkpoint is missing or incompatible."
    )

# Stage 2 initialization always begins from Stage 1 best weights.
frozen_checkpoint = torch.load(
    FROZEN_BEST,
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(
    frozen_checkpoint[
        "model_state_dict"
    ]
)

model.to(
    DEVICE
)

for parameter in (
    model.parameters()
):
    parameter.requires_grad = True

finetune_trainable = sum(
    parameter.numel()
    for parameter
    in model.parameters()
    if parameter.requires_grad
)

print(
    "Trainable params (fine-tune):",
    f"{finetune_trainable:,}",
)

# Resume Stage 2 only if a compatible Stage 2 checkpoint from THIS notebook/run exists.
finetune_summary = train_stage(
    model=model,
    stage_name="finetune",
    epochs=(
        CONFIG.finetune_epochs
    ),
    lr=(
        CONFIG.finetune_lr
    ),
    stage_dir=(
        FINETUNE_DIR
    ),
    frozen_backbone=False,
    allow_resume=True,
)

atomic_json_dump(
    finetune_summary,
    DIRS["metrics"]
    / "finetune_training_summary.json",
)

finetune_summary


Trainable params (fine-tune): 4,008,829


finetune train 1/15:   0%|          | 0/49 [00:00<?, ?it/s]

finetune val 1/15:   0%|          | 0/7 [00:00<?, ?it/s]

[finetune] epoch 01/15 | train_loss=0.74692 | val_loss=0.71918 | val_auc=0.53494 | val_f1=0.30657 | lr=2.00e-05 | time=14.6s


finetune train 2/15:   0%|          | 0/49 [00:00<?, ?it/s]

...

[finetune] epoch 14/15 | train_loss=0.46060 | val_loss=0.67852 | val_auc=0.66086 | val_f1=0.54945 | lr=2.00e-05 | time=15.7s


finetune train 15/15:   0%|          | 0/49 [00:00<?, ?it/s]

finetune val 15/15:   0%|          | 0/7 [00:00<?, ?it/s]

[finetune] epoch 15/15 | train_loss=0.44790 | val_loss=0.66910 | val_auc=0.67857 | val_f1=0.56989 | lr=1.00e-05 | time=16.5s


{'stage': 'finetune',
 'history': [{'stage': 'finetune',
   'epoch': 1,
   'lr': 2e-05,
   'train_loss': 0.7469199963782112,
   'val_loss': 0.7191818485073015,
   'train_threshold': 0.5,
   'train_accuracy': 0.5108834827144686,
   'train_balanced_accuracy': 0.5084697410875896,
   'train_precision': 0.53125,
   'train_recall': 0.13144329896907217,
   'train_f1': 0.21074380165289255,
   'train_specificity': 0.8854961832061069,
   'train_roc_auc': 0.535649641929645,
   'train_average_precision': 0.5271115027836863,
   'train_tn': 696,
   'train_fp': 90,
   'train_fn': 674,
   'train_tp': 102,
   'val_threshold': 0.5,
   'val_accuracy': 0.5343137254901961,
   'val_balanced_accuracy': 0.5222371967654986,
   'val_precision': 0.5384615384615384,
   'val_recall': 0.21428571428571427,
   'val_f1': 0.30656934306569344,
   'val_specificity': 0.8301886792452831,
   'val_roc_auc': 0.5349441663457836,
   'val_average_precision': 0.5175539896258295,
   'val_tn': 88,
   'val_fp': 18,
   'val_fn': 77,


In [21]:

# ============================================================
# CELL 20 — VALIDATION THRESHOLD SELECTION
# ============================================================

FINAL_BEST_CHECKPOINT = (
    FINETUNE_DIR
    / "best.ckpt"
)

if not checkpoint_is_compatible(
    FINAL_BEST_CHECKPOINT,
    "finetune",
):
    raise RuntimeError(
        "Final best fine-tune checkpoint is missing or incompatible."
    )

final_checkpoint = torch.load(
    FINAL_BEST_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

model.load_state_dict(
    final_checkpoint[
        "model_state_dict"
    ]
)

model.to(
    DEVICE
)

model.eval()

validation_output = run_epoch(
    model,
    loaders["val"],
    training=False,
    description="Final validation",
)

best_threshold, threshold_table = (
    select_validation_threshold(
        validation_output[
            "labels"
        ],
        validation_output[
            "probabilities"
        ],
    )
)

validation_selected_metrics = (
    binary_metrics(
        validation_output[
            "labels"
        ],
        validation_output[
            "probabilities"
        ],
        threshold=(
            best_threshold
        ),
    )
)

atomic_csv_dump(
    threshold_table,
    DIRS["metrics"]
    / "validation_threshold_search.csv",
)

atomic_json_dump(
    validation_selected_metrics,
    DIRS["metrics"]
    / "validation_selected_threshold_metrics.json",
)

print(
    "Validation-selected threshold:",
    best_threshold,
)

print(
    json.dumps(
        validation_selected_metrics,
        indent=2,
    )
)


Final validation:   0%|          | 0/7 [00:00<?, ?it/s]

Validation-selected threshold: 0.27999999999999997
{
  "threshold": 0.27999999999999997,
  "accuracy": 0.5931372549019608,
  "balanced_accuracy": 0.6042549095109742,
  "precision": 0.5471698113207547,
  "recall": 0.8877551020408163,
  "f1": 0.6770428015564203,
  "specificity": 0.32075471698113206,
  "roc_auc": 0.6819407008086252,
  "average_precision": 0.6976401318715859,
  "tn": 34,
  "fp": 72,
  "fn": 11,
  "tp": 87
}


In [22]:

# ============================================================
# CELL 21 — FINAL TEST EVALUATION
# ============================================================

test_output = run_epoch(
    model,
    loaders["test"],
    training=False,
    description="Final test",
)

test_metrics = binary_metrics(
    test_output[
        "labels"
    ],
    test_output[
        "probabilities"
    ],
    threshold=(
        best_threshold
    ),
)

test_predictions = pd.DataFrame({
    "sample_id": (
        test_output[
            "sample_ids"
        ]
    ),
    "path": (
        test_output[
            "paths"
        ]
    ),
    "label_int": (
        test_output[
            "labels"
        ]
    ),
    "label": [
        INT_TO_LABEL[
            int(value)
        ]
        for value
        in test_output[
            "labels"
        ]
    ],
    "prob_fake": (
        test_output[
            "probabilities"
        ]
    ),
})

test_predictions[
    "threshold"
] = (
    best_threshold
)

test_predictions[
    "pred_int"
] = (
    test_predictions[
        "prob_fake"
    ].to_numpy()
    >= best_threshold
).astype(int)

test_predictions[
    "prediction"
] = (
    test_predictions[
        "pred_int"
    ]
    .map(
        INT_TO_LABEL
    )
)

test_predictions[
    "correct"
] = (
    test_predictions[
        "label_int"
    ]
    == test_predictions[
        "pred_int"
    ]
)

atomic_csv_dump(
    pd.DataFrame([
        {
            "model": "EfficientNet-B0",
            **test_metrics,
        }
    ]),
    DIRS["metrics"]
    / "final_test_metrics.csv",
)

atomic_csv_dump(
    test_predictions,
    DIRS["predictions"]
    / "test_predictions.csv",
)

print(
    json.dumps(
        test_metrics,
        indent=2,
    )
)


Final test:   0%|          | 0/7 [00:00<?, ?it/s]

{
  "threshold": 0.27999999999999997,
  "accuracy": 0.5459183673469388,
  "balanced_accuracy": 0.5556539864512767,
  "precision": 0.51875,
  "recall": 0.8736842105263158,
  "f1": 0.6509803921568628,
  "specificity": 0.2376237623762376,
  "roc_auc": 0.5873892652423137,
  "average_precision": 0.5761101304032796,
  "tn": 24,
  "fp": 77,
  "fn": 12,
  "tp": 83
}


In [23]:
# ============================================================
# CELL 22 — FIGURES
# ============================================================

def save_figure(
    figure,
    filename: str,
) -> Path:
    path = (
        DIRS["figures"]
        / filename
    )

    figure.savefig(
        path,
        dpi=150,
        bbox_inches="tight",
    )

    plt.close(
        figure
    )

    with Image.open(
        path
    ) as image:
        if min(
            image.size
        ) < 600:
            raise RuntimeError(
                "Figure resolution quality gate failed: "
                f"{path} -> {image.size}"
            )

    return path


frozen_history = pd.read_csv(
    DIRS["metrics"]
    / "frozen_training_history.csv"
)

finetune_history = pd.read_csv(
    DIRS["metrics"]
    / "finetune_training_history.csv"
)

history = pd.concat(
    [
        frozen_history,
        finetune_history,
    ],
    ignore_index=True,
)

history[
    "global_epoch"
] = np.arange(
    1,
    len(history)
    + 1,
)

# Loss
figure, axis = plt.subplots(
    figsize=(10, 6),
    dpi=150,
)

axis.plot(
    history[
        "global_epoch"
    ],
    history[
        "train_loss"
    ],
    label="Training Loss",
)

axis.plot(
    history[
        "global_epoch"
    ],
    history[
        "val_loss"
    ],
    label="Validation Loss",
)

axis.set_title(
    "EfficientNet-B0 Eyebrow ROI Training and Validation Loss"
)

axis.set_xlabel(
    "Global Epoch"
)

axis.set_ylabel(
    "Loss"
)

axis.legend()
axis.grid(
    True,
    alpha=0.25,
)

figure.tight_layout()

save_figure(
    figure,
    "training_validation_loss_curve.png",
)

# Validation metrics
figure, axis = plt.subplots(
    figsize=(10, 6),
    dpi=150,
)

axis.plot(
    history[
        "global_epoch"
    ],
    history[
        "val_roc_auc"
    ],
    label="Validation ROC-AUC",
)

axis.plot(
    history[
        "global_epoch"
    ],
    history[
        "val_f1"
    ],
    label="Validation F1",
)

axis.plot(
    history[
        "global_epoch"
    ],
    history[
        "val_balanced_accuracy"
    ],
    label="Validation Balanced Accuracy",
)

axis.set_title(
    "EfficientNet-B0 Eyebrow ROI Validation Metrics"
)

axis.set_xlabel(
    "Global Epoch"
)

axis.set_ylabel(
    "Score"
)

axis.set_ylim(
    0,
    1,
)

axis.legend()
axis.grid(
    True,
    alpha=0.25,
)

figure.tight_layout()

save_figure(
    figure,
    "validation_metrics_curve.png",
)

y_test = test_output[
    "labels"
]

p_test = test_output[
    "probabilities"
]

# ROC
fpr, tpr, _ = roc_curve(
    y_test,
    p_test,
)

roc_auc = roc_auc_score(
    y_test,
    p_test,
)

figure, axis = plt.subplots(
    figsize=(10, 6),
    dpi=150,
)

axis.plot(
    fpr,
    tpr,
    label=f"Eyebrow EfficientNet-B0 (AUC={roc_auc:.3f})",
)

axis.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Chance",
)

axis.set_title(
    "Test ROC Curve"
)

axis.set_xlabel(
    "False Positive Rate"
)

axis.set_ylabel(
    "True Positive Rate"
)

axis.legend()
axis.grid(
    True,
    alpha=0.25,
)

figure.tight_layout()

save_figure(
    figure,
    "test_roc_curve.png",
)

# PR
precision_curve, recall_curve, _ = (
    precision_recall_curve(
        y_test,
        p_test,
    )
)

average_precision = (
    average_precision_score(
        y_test,
        p_test,
    )
)

figure, axis = plt.subplots(
    figsize=(10, 6),
    dpi=150,
)

axis.plot(
    recall_curve,
    precision_curve,
    label=(
        "Eyebrow EfficientNet-B0 "
        f"(AP={average_precision:.3f})"
    ),
)

axis.set_title(
    "Test Precision-Recall Curve"
)

axis.set_xlabel(
    "Recall"
)

axis.set_ylabel(
    "Precision"
)

axis.legend()
axis.grid(
    True,
    alpha=0.25,
)

figure.tight_layout()

save_figure(
    figure,
    "test_precision_recall_curve.png",
)

# Confusion matrix
matrix = confusion_matrix(
    y_test,
    (
        p_test
        >= best_threshold
    ).astype(int),
    labels=[
        0,
        1,
    ],
)

figure, axis = plt.subplots(
    figsize=(8, 8),
    dpi=150,
)

image = axis.imshow(
    matrix
)

figure.colorbar(
    image,
    ax=axis,
)

axis.set_title(
    f"Test Confusion Matrix (Threshold={best_threshold:.3f})"
)

axis.set_xlabel(
    "Predicted Label"
)

axis.set_ylabel(
    "True Label"
)

axis.set_xticks(
    [
        0,
        1,
    ],
    labels=[
        "Real",
        "Fake",
    ],
)

axis.set_yticks(
    [
        0,
        1,
    ],
    labels=[
        "Real",
        "Fake",
    ],
)

for row_index in range(
    2
):
    for column_index in range(
        2
    ):
        axis.text(
            column_index,
            row_index,
            str(
                matrix[
                    row_index,
                    column_index,
                ]
            ),
            ha="center",
            va="center",
            fontsize=14,
        )

figure.tight_layout()

save_figure(
    figure,
    "test_confusion_matrix.png",
)

# Validation threshold analysis
figure, axis = plt.subplots(
    figsize=(10, 6),
    dpi=150,
)

axis.plot(
    threshold_table[
        "threshold"
    ],
    threshold_table[
        "f1"
    ],
    label="Validation F1",
)

axis.plot(
    threshold_table[
        "threshold"
    ],
    threshold_table[
        "balanced_accuracy"
    ],
    label="Validation Balanced Accuracy",
)

axis.plot(
    threshold_table[
        "threshold"
    ],
    threshold_table[
        "specificity"
    ],
    label="Validation Specificity",
)

axis.axvline(
    best_threshold,
    linestyle="--",
    label=f"Selected={best_threshold:.3f}",
)

axis.set_title(
    "Validation Threshold Analysis"
)

axis.set_xlabel(
    "Threshold"
)

axis.set_ylabel(
    "Score"
)

axis.set_ylim(
    0,
    1,
)

axis.legend()
axis.grid(
    True,
    alpha=0.25,
)

figure.tight_layout()

save_figure(
    figure,
    "validation_threshold_analysis.png",
)

print(
    "Figures saved:",
    DIRS["figures"],
)


Figures saved: /content/drive/MyDrive/AISC DeepFake Çalışmaları/Deneyler/Nazlıcan/Deney 1/Sonuçlar/20260808_1248_eyebrow_efficientnet_b0_seed42/figures


In [24]:

# ============================================================
# CELL 23 — RELOAD / INFERENCE QUALITY GATE
# ============================================================

def inference_reload_test() -> dict:
    reloaded_model = (
        build_model(
            pretrained=False
        )
        .to(DEVICE)
    )

    checkpoint = torch.load(
        FINAL_BEST_CHECKPOINT,
        map_location="cpu",
        weights_only=False,
    )

    reloaded_model.load_state_dict(
        checkpoint[
            "model_state_dict"
        ]
    )

    reloaded_model.to(
        DEVICE
    )

    reloaded_model.eval()

    batch = next(
        iter(
            loaders["test"]
        )
    )

    images = (
        batch["image"][
            : min(
                4,
                len(
                    batch["image"]
                ),
            )
        ]
        .to(DEVICE)
    )

    with torch.no_grad():
        probabilities = torch.sigmoid(
            reloaded_model(
                images
            )
            .squeeze(1)
            .float()
        )

    if probabilities.ndim != 1:
        raise RuntimeError(
            "Unexpected inference output shape: "
            f"{tuple(probabilities.shape)}"
        )

    if not torch.isfinite(
        probabilities
    ).all():
        raise FloatingPointError(
            "Reloaded model produced non-finite probabilities."
        )

    if (
        (
            probabilities
            < 0
        )
        | (
            probabilities
            > 1
        )
    ).any():
        raise RuntimeError(
            "Reloaded model produced probabilities outside [0, 1]."
        )

    return {
        "status": "PASSED",
        "checkpoint": str(
            FINAL_BEST_CHECKPOINT
        ),
        "n_samples": int(
            len(
                probabilities
            )
        ),
        "probabilities": (
            probabilities
            .detach()
            .cpu()
            .numpy()
            .astype(float)
            .tolist()
        ),
    }


inference_test = (
    inference_reload_test()
)

atomic_json_dump(
    inference_test,
    DIRS["metrics"]
    / "inference_reload_test.json",
)

print(
    json.dumps(
        inference_test,
        indent=2,
    )
)


{
  "status": "PASSED",
  "checkpoint": "/content/drive/MyDrive/AISC DeepFake \u00c7al\u0131\u015fmalar\u0131/Deneyler/Nazl\u0131can/Deney 1/Sonu\u00e7lar/20260808_1248_eyebrow_efficientnet_b0_seed42/checkpoints/finetune/best.ckpt",
  "n_samples": 4,
  "probabilities": [
    0.5050852298736572,
    0.5161343216896057,
    0.29075413942337036,
    0.44651493430137634
  ]
}


In [ ]:
# ============================================================
# CELL 24 — FINAL MANIFEST + RUN SUMMARY + COMPLETE MARKER
# ============================================================

def build_output_manifest(
    root: Path,
) -> pd.DataFrame:
    rows = []

    for path in sorted(
        root.rglob("*")
    ):
        if not path.is_file():
            continue

        # Do not hash/include the manifest itself recursively.
        if path.name == "output_manifest.csv":
            continue

        # Exclude temporary files if any.
        if path.name.endswith(
            ".tmp"
        ):
            continue

        size = path.stat().st_size

        digest = (
            sha256_file(
                path
            )
            if size
            <= 50
            * 1024
            * 1024
            else ""
        )

        rows.append({
            "relative_path": str(
                path.relative_to(
                    root
                )
            ),
            "size_bytes": int(
                size
            ),
            "sha256": digest,
        })

    return pd.DataFrame(
        rows
    )


run_summary = {
    "run_id": RUN_ID,
    "run_schema_version": (
        RUN_SCHEMA_VERSION
    ),
    "experiment": (
        "EfficientNet-B0 Eyebrow ROI Transfer Learning Baseline"
    ),
    "model": (
        "torchvision EfficientNet-B0"
    ),
    "pretrained_weights": (
        "EfficientNet_B0_Weights.DEFAULT"
    ),
    "input": (
        "raw color eyebrow ROI crops"
    ),
    "preprocessing": (
        "aspect-ratio preserving resize + ImageNet-mean padding "
        "+ ImageNet pretrained normalization"
    ),
    "image_size": (
        CONFIG.image_size
    ),
    "seed": (
        CONFIG.seed
    ),
    "device": str(
        DEVICE
    ),
    "metadata_path": str(
        METADATA_PATH
    ),
    "eyebrow_root": str(
        KAS_ROOT
    ),
    "results_root": str(
        RESULTS_ROOT
    ),
    "run_dir": str(
        RUN_DIR
    ),
    "split_counts": (
        data_accounting[
            "split_class_counts"
        ]
    ),
    "source_group_leakage_status": (
        data_accounting[
            "source_group_leakage_status"
        ]
    ),
    "source_group_column": (
        data_accounting[
            "source_group_column"
        ]
    ),
    "validation_selected_threshold": (
        best_threshold
    ),
    "validation_metrics": (
        validation_selected_metrics
    ),
    "final_test_metrics": (
        test_metrics
    ),
    "inference_reload_test": (
        inference_test[
            "status"
        ]
    ),
    "output_file_count": None,
}

# Write a provisional summary + completion marker first.
atomic_json_dump(
    run_summary,
    RUN_DIR
    / "run_summary.json",
)

atomic_json_dump(
    {
        "status": "COMPLETE",
        "run_id": RUN_ID,
        "completed_at": (
            datetime.now()
            .isoformat()
        ),
    },
    RUN_DIR
    / RUN_COMPLETE_FILE,
)

# Build manifest after all key files exist, then update summary count.
manifest = build_output_manifest(
    RUN_DIR
)

run_summary[
    "output_file_count"
] = int(
    len(
        manifest
    )
)

atomic_json_dump(
    run_summary,
    RUN_DIR
    / "run_summary.json",
)

# Rebuild once because run_summary changed.
manifest = build_output_manifest(
    RUN_DIR
)

atomic_csv_dump(
    manifest,
    RUN_DIR
    / "output_manifest.csv",
)

LOGGER.info(
    "Experiment complete: %s | output files=%d",
    RUN_ID,
    len(manifest),
)

print("=" * 80)
print("EXPERIMENT COMPLETE")
print("=" * 80)

print(
    json.dumps(
        run_summary,
        indent=2,
        ensure_ascii=False,
        default=str,
    )
)

print()
print("FINAL OUTPUT DIRECTORY:")
print(RUN_DIR)
